In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [3]:
MODEL_READY_CSV = Path("../data/model_ready/match_by_match.csv")

In [4]:
dataset = pd.read_csv(MODEL_READY_CSV)

In [5]:
dataset.shape

(1243, 31)

In [6]:
dataset.head()

,match_id,date,venue,team1,team2,toss_winner,toss_decision,winner,result,team1_score,...,team2_last_5_avg_score,team2_last_5_runs_conceded,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue
0,335982,2008-04-18,"M Chinnaswamy Stadium, Bengaluru",Kolkata Knight Riders,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,Kolkata Knight Riders,won by 140 runs,222,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,335983,2008-04-19,"Punjab Cricket Association IS Bindra Stadium, ...",Chennai Super Kings,Punjab Kings,Chennai Super Kings,bat,Chennai Super Kings,won by 33 runs,240,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,335984,2008-04-19,"Arun Jaitley Stadium, Delhi",Rajasthan Royals,Delhi Capitals,Rajasthan Royals,bat,Delhi Capitals,won by 9 wickets,129,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,335986,2008-04-20,"Eden Gardens, Kolkata",Deccan Chargers,Kolkata Knight Riders,Deccan Chargers,bat,Kolkata Knight Riders,won by 5 wickets,110,...,222.0,82.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,335985,2008-04-20,"Wankhede Stadium, Mumbai",Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Royal Challengers Bengaluru,won by 5 wickets,165,...,82.0,222.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
dataset.tail(5)

,match_id,date,venue,team1,team2,toss_winner,toss_decision,winner,result,team1_score,...,team2_last_5_avg_score,team2_last_5_runs_conceded,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue
1238,1529313,2026-05-24,"Eden Gardens, Kolkata",Delhi Capitals,Kolkata Knight Riders,Kolkata Knight Riders,field,Delhi Capitals,won by 40 runs,203,...,180.6,173.2,0.490,0.409,0.565,0.500,160.73,0.563,0.200,0.564
1239,1535462,2026-05-26,"Himachal Pradesh Cricket Association Stadium, ...",Royal Challengers Bengaluru,Gujarat Titans,Gujarat Titans,field,Royal Challengers Bengaluru,won by 92 runs,254,...,202.2,157.6,0.550,0.486,0.690,0.500,172.81,0.438,0.667,0.000
1240,1535463,2026-05-27,Maharaja Yadavindra Singh International Cricke...,Rajasthan Royals,Sunrisers Hyderabad,Sunrisers Hyderabad,field,Rajasthan Royals,won by 47 runs,243,...,184.4,183.8,0.545,0.383,0.492,0.359,210.90,0.750,1.000,0.000
1241,1535464,2026-05-29,Maharaja Yadavindra Singh International Cricke...,Rajasthan Royals,Gujarat Titans,Rajasthan Royals,bat,Gujarat Titans,won by 7 wickets,214,...,201.2,175.8,0.545,0.383,0.667,0.500,212.33,0.600,1.000,0.000
1242,1535465,2026-05-31,"Narendra Modi Stadium, Ahmedabad",Gujarat Titans,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,Royal Challengers Bengaluru,won by 5 wickets,155,...,207.4,194.8,0.667,0.500,0.550,0.486,176.40,0.469,0.600,0.429


In [8]:
dataset["winner"].value_counts()

winner
Mumbai Indians                 155
Chennai Super Kings            148
Royal Challengers Bengaluru    143
Kolkata Knight Riders          140
Punjab Kings                   126
Delhi Capitals                 125
Rajasthan Royals               123
Sunrisers Hyderabad            102
Gujarat Titans                  47
Lucknow Super Giants            34
Deccan Chargers                 29
Rising Pune Supergiants         15
Gujarat Lions                   13
Pune Warriors                   12
Kochi Tuskers Kerala             6
Name: count, dtype: int64

In [9]:
dataset.isna().sum()

match_id                                0
date                                    0
venue                                   0
team1                                   0
team2                                   6
toss_winner                             0
toss_decision                           0
winner                                 25
result                                 25
team1_score                             0
team2_score                             0
team1_players                           0
team2_players                           0
team1_total_wins_against_team2          0
team2_total_wins_against_team1          0
team1_wins_against_team2_last_three     0
team2_wins_against_team1_last_three     0
team1_form_last_5                       0
team2_form_last_5                       0
team1_last_5_avg_score                  0
team1_last_5_runs_conceded              0
team2_last_5_avg_score                  0
team2_last_5_runs_conceded              0
team1_chasing_win_rate            

In [10]:
dataset = dataset.dropna(subset=['winner'])

dataset.shape

(1218, 31)

In [11]:
def get_winner_slot(row):
    if row['winner'] == row['team1']:
        return 1
    else:
        return 0

In [12]:
def get_toss_winner_slot(row):
    if row['toss_winner'] == row['team1']:
        return 1
    else:
        return 0

In [13]:
dataset['winner_slot'] = dataset.apply(get_winner_slot, axis=1)
dataset['toss_winner_slot'] = dataset.apply(get_toss_winner_slot, axis=1)

dataset.head(5)

,match_id,date,venue,team1,team2,toss_winner,toss_decision,winner,result,team1_score,...,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue,winner_slot,toss_winner_slot
0,335982,2008-04-18,"M Chinnaswamy Stadium, Bengaluru",Kolkata Knight Riders,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,Kolkata Knight Riders,won by 140 runs,222,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
1,335983,2008-04-19,"Punjab Cricket Association IS Bindra Stadium, ...",Chennai Super Kings,Punjab Kings,Chennai Super Kings,bat,Chennai Super Kings,won by 33 runs,240,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1
2,335984,2008-04-19,"Arun Jaitley Stadium, Delhi",Rajasthan Royals,Delhi Capitals,Rajasthan Royals,bat,Delhi Capitals,won by 9 wickets,129,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
3,335986,2008-04-20,"Eden Gardens, Kolkata",Deccan Chargers,Kolkata Knight Riders,Deccan Chargers,bat,Kolkata Knight Riders,won by 5 wickets,110,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
4,335985,2008-04-20,"Wankhede Stadium, Mumbai",Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Royal Challengers Bengaluru,won by 5 wickets,165,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


Saving copy of the dataset before the dropping of the essential columns which will be used during model training

In [14]:
dataset_pre_drop = dataset.copy()

Removing categorical features and teams' score before the model training

In [15]:
dataset = dataset.drop(["date", "venue", "result", "toss_winner",
                        "toss_decision", "team1", "team2",
                        "team1_score", 
                        "team2_score", 
                        "team1_players", 
                        "team2_players", "winner"], axis=1)

dataset = dataset.set_index("match_id")

dataset.head(5)

,team1_total_wins_against_team2,team2_total_wins_against_team1,team1_wins_against_team2_last_three,team2_wins_against_team1_last_three,team1_form_last_5,team2_form_last_5,team1_last_5_avg_score,team1_last_5_runs_conceded,team2_last_5_avg_score,team2_last_5_runs_conceded,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue,winner_slot,toss_winner_slot
match_id,,,,,,,,,,,,,,,,,,,,
335982,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
335983,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1
335984,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
335986,0,0,0,0,0,1,0.0,0.0,222.0,82.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
335985,0,0,0,0,0,0,0.0,0.0,82.0,222.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


We have removed the both the teams' scores because we didn't have information about their score before the match ever started.

Moving the `winner_slot` column to the front while keeping all other columns in their original order.

In [16]:
cols = list(dataset.columns)
cols.remove("winner_slot")
cols = ["winner_slot"] + cols
dataset = dataset[cols]
dataset.columns

Index(['winner_slot', 'team1_total_wins_against_team2',
       'team2_total_wins_against_team1', 'team1_wins_against_team2_last_three',
       'team2_wins_against_team1_last_three', 'team1_form_last_5',
       'team2_form_last_5', 'team1_last_5_avg_score',
       'team1_last_5_runs_conceded', 'team2_last_5_avg_score',
       'team2_last_5_runs_conceded', 'team1_chasing_win_rate',
       'team1_defending_win_rate', 'team2_chasing_win_rate',
       'team2_defending_win_rate', 'venue_avg_score', 'chasing_win_rate_venue',
       'team1_win_rate_at_venue', 'team2_win_rate_at_venue',
       'toss_winner_slot'],
      dtype='str')

In [17]:
dataset.head(5)

,winner_slot,team1_total_wins_against_team2,team2_total_wins_against_team1,team1_wins_against_team2_last_three,team2_wins_against_team1_last_three,team1_form_last_5,team2_form_last_5,team1_last_5_avg_score,team1_last_5_runs_conceded,team2_last_5_avg_score,team2_last_5_runs_conceded,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue,toss_winner_slot
match_id,,,,,,,,,,,,,,,,,,,,
335982,1,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
335983,1,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
335984,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
335986,0,0,0,0,0,0,1,0.0,0.0,222.0,82.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
335985,0,0,0,0,0,0,0,0.0,0.0,82.0,222.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


Setting the first column (`winner_slot`) as target and other features as input columns for the model training

In [18]:
target_value = [0]
print(dataset.columns[target_value])

Index(['winner_slot'], dtype='str')


In [19]:
input_values = range(1, dataset.columns.shape[0])

print(dataset.columns[input_values])

Index(['team1_total_wins_against_team2', 'team2_total_wins_against_team1',
       'team1_wins_against_team2_last_three',
       'team2_wins_against_team1_last_three', 'team1_form_last_5',
       'team2_form_last_5', 'team1_last_5_avg_score',
       'team1_last_5_runs_conceded', 'team2_last_5_avg_score',
       'team2_last_5_runs_conceded', 'team1_chasing_win_rate',
       'team1_defending_win_rate', 'team2_chasing_win_rate',
       'team2_defending_win_rate', 'venue_avg_score', 'chasing_win_rate_venue',
       'team1_win_rate_at_venue', 'team2_win_rate_at_venue',
       'toss_winner_slot'],
      dtype='str')


### Splitting the dataset into train and test

In [20]:
x = dataset.iloc[:, input_values]
y = dataset.iloc[:, target_value]

X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2, random_state=42)

### Training and saving machine learning models.

In [21]:
pipeline = Pipeline([
    ("classifier", DecisionTreeClassifier())
])

pipeline.fit(X_train, Y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(Y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/no_score/decision_tree.pkl")

Accuracy: 0.5164
Classification Report:
               precision    recall  f1-score   support

           0       0.57      0.53      0.55       135
           1       0.46      0.50      0.48       109

    accuracy                           0.52       244
   macro avg       0.51      0.51      0.51       244
weighted avg       0.52      0.52      0.52       244

Confusion Matrix:
 [[72 63]
 [55 54]]


['../models/no_score/decision_tree.pkl']

In [22]:
pipeline = Pipeline([
    ("classifier", RandomForestClassifier())
])

pipeline.fit(X_train, Y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(Y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/no_score/random_forest.pkl")

c:\venvs\datasci\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Accuracy: 0.4836
Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.69      0.60       135
           1       0.37      0.23      0.28       109

    accuracy                           0.48       244
   macro avg       0.45      0.46      0.44       244
weighted avg       0.46      0.48      0.46       244

Confusion Matrix:
 [[93 42]
 [84 25]]


['../models/no_score/random_forest.pkl']

In [23]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifer", LogisticRegression())
])

pipeline.fit(X_train, Y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(Y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/no_score/logistic_regression.pkl")

Accuracy: 0.5000
Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.76      0.63       135
           1       0.38      0.18      0.25       109

    accuracy                           0.50       244
   macro avg       0.46      0.47      0.44       244
weighted avg       0.46      0.50      0.46       244

Confusion Matrix:
 [[102  33]
 [ 89  20]]


c:\venvs\datasci\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['../models/no_score/logistic_regression.pkl']

In [24]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

pipeline.fit(X_train, Y_train)

preds_continuous = pipeline.predict(X_test)

predictions = np.round(preds_continuous).astype(int)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print(classification_report(Y_test, predictions))
print(confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/no_score/linear_regression.pkl")

Accuracy: 0.4959
              precision    recall  f1-score   support

           0       0.53      0.75      0.62       135
           1       0.37      0.18      0.25       109

    accuracy                           0.50       244
   macro avg       0.45      0.47      0.43       244
weighted avg       0.46      0.50      0.45       244

[[101  34]
 [ 89  20]]


['../models/no_score/linear_regression.pkl']

In [25]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifer", SVC(kernel='rbf'))
])

pipeline.fit(X_train, Y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(Y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/no_score/support_vector_classifier.pkl")

Accuracy: 0.5041
Classification Report:
               precision    recall  f1-score   support

           0       0.54      0.76      0.63       135
           1       0.39      0.19      0.26       109

    accuracy                           0.50       244
   macro avg       0.46      0.47      0.44       244
weighted avg       0.47      0.50      0.46       244

Confusion Matrix:
 [[102  33]
 [ 88  21]]


c:\venvs\datasci\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['../models/no_score/support_vector_classifier.pkl']

In [26]:
pipeline = Pipeline([
    ("classifier", XGBClassifier())
])

pipeline.fit(X_train, Y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(Y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/no_score/xg_boost.pkl")

Accuracy: 0.5123
Classification Report:
               precision    recall  f1-score   support

           0       0.55      0.63      0.59       135
           1       0.44      0.37      0.40       109

    accuracy                           0.51       244
   macro avg       0.50      0.50      0.50       244
weighted avg       0.50      0.51      0.51       244

Confusion Matrix:
 [[85 50]
 [69 40]]


['../models/no_score/xg_boost.pkl']

We have not dropped the team1 score in this case because we are training the model by using first innings' score as the feature

In [27]:
dataset = dataset_pre_drop.copy()

dataset = dataset.drop(["date", "venue", "result", "toss_winner",
                        "toss_decision", "team1", "team2",
                        "team2_score", 
                        "team1_players", 
                        "team2_players", "winner"], axis=1)

dataset = dataset.set_index("match_id")

dataset.head(5)

,team1_score,team1_total_wins_against_team2,team2_total_wins_against_team1,team1_wins_against_team2_last_three,team2_wins_against_team1_last_three,team1_form_last_5,team2_form_last_5,team1_last_5_avg_score,team1_last_5_runs_conceded,team2_last_5_avg_score,...,team1_chasing_win_rate,team1_defending_win_rate,team2_chasing_win_rate,team2_defending_win_rate,venue_avg_score,chasing_win_rate_venue,team1_win_rate_at_venue,team2_win_rate_at_venue,winner_slot,toss_winner_slot
match_id,,,,,,,,,,,,,,,,,,,,,
335982,222,0,0,0,0,0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
335983,240,0,0,0,0,0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1
335984,129,0,0,0,0,0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
335986,110,0,0,0,0,0,1,0.0,0.0,222.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
335985,165,0,0,0,0,0,0,0.0,0.0,82.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


In [28]:
cols = list(dataset.columns)
cols.remove("winner_slot")
cols = ["winner_slot"] + cols
dataset = dataset[cols]
dataset.columns

Index(['winner_slot', 'team1_score', 'team1_total_wins_against_team2',
       'team2_total_wins_against_team1', 'team1_wins_against_team2_last_three',
       'team2_wins_against_team1_last_three', 'team1_form_last_5',
       'team2_form_last_5', 'team1_last_5_avg_score',
       'team1_last_5_runs_conceded', 'team2_last_5_avg_score',
       'team2_last_5_runs_conceded', 'team1_chasing_win_rate',
       'team1_defending_win_rate', 'team2_chasing_win_rate',
       'team2_defending_win_rate', 'venue_avg_score', 'chasing_win_rate_venue',
       'team1_win_rate_at_venue', 'team2_win_rate_at_venue',
       'toss_winner_slot'],
      dtype='str')

### Splitting the dataset into train and test. But this time we have added new feature `team1_score`.

In [29]:
x = dataset.iloc[:, input_values]
y = dataset.iloc[:, target_value]

X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2, random_state=42)

### Training and saving machine learning models.

In [30]:
pipeline = Pipeline([
    ("classifier", DecisionTreeClassifier())
])

pipeline.fit(X_train, Y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(Y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/with_score/decision_tree.pkl")

Accuracy: 0.5738
Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.60      0.61       135
           1       0.52      0.54      0.53       109

    accuracy                           0.57       244
   macro avg       0.57      0.57      0.57       244
weighted avg       0.58      0.57      0.57       244

Confusion Matrix:
 [[81 54]
 [50 59]]


['../models/with_score/decision_tree.pkl']

In [31]:
pipeline = Pipeline([
    ("classifier", RandomForestClassifier())
])

pipeline.fit(X_train, Y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(Y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/with_score/random_forest.pkl")

c:\venvs\datasci\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Accuracy: 0.7090
Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.77      0.75       135
           1       0.69      0.63      0.66       109

    accuracy                           0.71       244
   macro avg       0.71      0.70      0.70       244
weighted avg       0.71      0.71      0.71       244

Confusion Matrix:
 [[104  31]
 [ 40  69]]


['../models/with_score/random_forest.pkl']

In [32]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifer", LogisticRegression())
])

pipeline.fit(X_train, Y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(Y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/with_score/logistic_regression.pkl")

Accuracy: 0.7008
Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.79      0.75       135
           1       0.70      0.59      0.64       109

    accuracy                           0.70       244
   macro avg       0.70      0.69      0.69       244
weighted avg       0.70      0.70      0.70       244

Confusion Matrix:
 [[107  28]
 [ 45  64]]


c:\venvs\datasci\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['../models/with_score/logistic_regression.pkl']

In [33]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

pipeline.fit(X_train, Y_train)

preds_continuous = pipeline.predict(X_test)

predictions = np.round(preds_continuous).astype(int)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print(classification_report(Y_test, predictions))
print(confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/with_score/linear_regression.pkl")

Accuracy: 0.7090
              precision    recall  f1-score   support

           0       0.71      0.79      0.75       135
           1       0.70      0.61      0.65       109

    accuracy                           0.71       244
   macro avg       0.71      0.70      0.70       244
weighted avg       0.71      0.71      0.71       244

[[107  28]
 [ 43  66]]


['../models/with_score/linear_regression.pkl']

In [34]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifer", SVC(kernel='rbf'))
])

pipeline.fit(X_train, Y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(Y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/with_score/support_vector_classifier.pkl")

Accuracy: 0.7090
Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.80      0.75       135
           1       0.71      0.60      0.65       109

    accuracy                           0.71       244
   macro avg       0.71      0.70      0.70       244
weighted avg       0.71      0.71      0.71       244

Confusion Matrix:
 [[108  27]
 [ 44  65]]


c:\venvs\datasci\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['../models/with_score/support_vector_classifier.pkl']

In [35]:
pipeline = Pipeline([
    ("classifier", XGBClassifier())
])

pipeline.fit(X_train, Y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(Y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(Y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(Y_test, predictions))

joblib.dump(pipeline, "../models/with_score/xg_boost.pkl")

Accuracy: 0.6762
Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.73      0.71       135
           1       0.64      0.61      0.63       109

    accuracy                           0.68       244
   macro avg       0.67      0.67      0.67       244
weighted avg       0.68      0.68      0.68       244

Confusion Matrix:
 [[98 37]
 [42 67]]


['../models/with_score/xg_boost.pkl']